1. orders.csv информация о заказах
•order_id – идентификатор заказа
•order_date – дата заказа (YYYY-MM-DD)
•delivery_date – дата доставки (YYYY-MM-DD)
•product_id – идентификатор товара (ссылка на product_info)
•quantity – количество единиц (1–50)
•price – цена за единицу (рубли)
•region – регион доставки (Центральный, Восточный, Западный, Северный, Южный)

2. product_info.csv справочник товаров
•product_id – идентификатор товара
•product_name – название товара
•category – категория (Электроника, Мебель, Канцтовары, Книги, Одежда)
•supplier – поставщик (несколько вариантов)
•base_stock – остаток на складе (число)

1.Загрузка и первичный просмотр: Загрузите оба CSV-файла в DataFrame. Выведите первые 5 строк каждого.

In [92]:
import pandas as pd
import numpy as np

orders = pd.read_csv("python/orders.csv", parse_dates=["order_date", "delivery_date"])
product_info = pd.read_csv("python/product_info.csv")

print(f"orders.csv:")
print(orders.head(5))
print(f"Размер: {orders.shape}")

print("\nproduct_info.csv")
print(product_info.head(5))
print(f"Размер: {products.shape}")

orders.csv:
   order_id order_date delivery_date  product_id  quantity  price       region
0         1 2023-01-10    2023-01-15           1         2  50000  Центральный
1         2 2023-01-12    2023-01-18           2         5  30000    Восточный
2         3 2023-01-14    2023-01-20           3        10    500     Западный
3         4 2023-01-15    2023-01-19           4        15   2000    Восточный
4         5 2023-01-18    2023-01-25           5         3   8000  Центральный
Размер: (200, 7)

product_info.csv
   product_id product_name     category     supplier  base_stock
0           1      Ноутбук  Электроника  Поставщик А         120
1           2     Смартфон  Электроника  Поставщик Б         300
2           3        Книга        Книги  Поставщик В          50
3           4     Наушники  Электроника  Поставщик Г         200
4           5         Стул       Мебель  Поставщик А          45
Размер: (20, 5)


2.Фильтрация: Отфильтруйте заказы, где region = "Восточный" и quantity > 10. Выведите количество таких заказов и первые 5 строк.

In [107]:
filtered = orders[(orders["region"] == "Восточный") & (orders["quantity"] > 10)]

print(f"Количество подходящих заказов: {len(filtered)}")
filtered.head()

Количество подходящих заказов: 17


,order_id,order_date,delivery_date,product_id,quantity,price,region
3,4,2023-01-15,2023-01-19,4,15,2000,Восточный
5,6,2023-01-20,2023-01-24,6,100,50,Восточный
24,25,2023-02-09,2023-02-17,15,20,150,Восточный
41,42,2023-02-26,2023-03-06,12,15,2800,Восточный
61,62,2023-03-18,2023-03-28,12,12,2900,Восточный


3.Объединение таблиц: Соедините orders и product_info по ключу product_id. В результирующем наборе должны быть все поля из обеих таблиц. Добавьте вычисляемый столбец total_cost = quantity * price.

In [ ]:
merged = orders.merge(product_info, on="product_id", how="left")

merged["total_cost"] = merged["quantity"] * merged["price"]

print(f"Размер объединённой таблицы: {merged.shape}")
merged.head()

Размер объединённой таблицы: (200, 12)


,order_id,order_date,delivery_date,product_id,quantity,price,region,product_name,category,supplier,base_stock,total_cost
0,1,2023-01-10,2023-01-15,1,2,50000,Центральный,Ноутбук,Электроника,Поставщик А,120,100000
1,2,2023-01-12,2023-01-18,2,5,30000,Восточный,Смартфон,Электроника,Поставщик Б,300,150000
2,3,2023-01-14,2023-01-20,3,10,500,Западный,Книга,Книги,Поставщик В,50,5000
3,4,2023-01-15,2023-01-19,4,15,2000,Восточный,Наушники,Электроника,Поставщик Г,200,30000
4,5,2023-01-18,2023-01-25,5,3,8000,Центральный,Стул,Мебель,Поставщик А,45,24000


4.Расчёт рабочих дней между датами: Для каждой строки вычислите количество рабочих дней (без учёта выходных и праздничных дней в России) между order_date и delivery_date. Добавьте столбец business_days.

In [112]:
import holidays

# ключи словаря {дата: название праздника}
ru_holidays = holidays.Russia(years=range(
    orders["order_date"].min().year,
    orders["delivery_date"].max().year + 1
)).keys()

# numpy.busday_count принимает массив дат в формате datetime64[D]
holiday_dates = np.array(list(ru_holidays), dtype="datetime64[D]")

# рабочие дни между датами заказа и доставки без выходных и праздников
merged["business_days"] = np.busday_count(
    merged["order_date"].to_numpy(dtype="datetime64[D]"),
    merged["delivery_date"].to_numpy(dtype="datetime64[D]"),
    holidays=holiday_dates
)

merged[["order_id", "order_date", "delivery_date", "business_days"]].head()

,order_id,order_date,delivery_date,business_days
0,1,2023-01-10,2023-01-15,4
1,2,2023-01-12,2023-01-18,4
2,3,2023-01-14,2023-01-20,4
3,4,2023-01-15,2023-01-19,3
4,5,2023-01-18,2023-01-25,5


In [113]:
print("\nСводка по business_days:")
merged["business_days"].describe().round(2)


Сводка по business_days:


count    200.00
mean       6.02
std        1.43
min        2.00
25%        5.00
50%        6.00
75%        7.00
max        9.00
Name: business_days, dtype: float64

5.Агрегация по регионам: Сгруппируйте данные по полю region и вычислите:
•среднее количество рабочих дней доставки (avg_business_days_by_region)
•общую сумму продаж (total_sales_by_region)
•количество заказов (order_count_by_region)
Результат отсортируйте по убыванию avg_business_days_by_region.

In [118]:
agg = (
    merged
    .groupby("region")
    .agg(
        avg_business_days_by_region=("business_days", "mean"),
        total_sales_by_region=("total_cost", "sum"),
        order_count_by_region=("order_id", "count"),
    )
).round(2).reset_index()
agg

,region,avg_business_days_by_region,total_sales_by_region,order_count_by_region
0,Восточный,5.91,5165800,43
1,Западный,5.95,6097345,40
2,Северный,6.11,794176,38
3,Центральный,5.98,6860550,41
4,Южный,6.18,3303480,38


6.Вывод результатов: Выведите итоговые таблицы из пп. 5 и 6 в удобном для чтения виде. (видимо опечатка, пукнкты 4 и 5)

In [115]:
# пункт 4
print("Объединение orders и product_info с business_days")
merged.head()

Объединение orders и product_info с business_days


,order_id,order_date,delivery_date,product_id,quantity,price,region,product_name,category,supplier,base_stock,total_cost,business_days
0,1,2023-01-10,2023-01-15,1,2,50000,Центральный,Ноутбук,Электроника,Поставщик А,120,100000,4
1,2,2023-01-12,2023-01-18,2,5,30000,Восточный,Смартфон,Электроника,Поставщик Б,300,150000,4
2,3,2023-01-14,2023-01-20,3,10,500,Западный,Книга,Книги,Поставщик В,50,5000,4
3,4,2023-01-15,2023-01-19,4,15,2000,Восточный,Наушники,Электроника,Поставщик Г,200,30000,3
4,5,2023-01-18,2023-01-25,5,3,8000,Центральный,Стул,Мебель,Поставщик А,45,24000,5


In [119]:
# пункт 5 (ранее вывела, дублирую)
print("Агрегация по регионам:")
agg

Агрегация по регионам:


,region,avg_business_days_by_region,total_sales_by_region,order_count_by_region
0,Восточный,5.91,5165800,43
1,Западный,5.95,6097345,40
2,Северный,6.11,794176,38
3,Центральный,5.98,6860550,41
4,Южный,6.18,3303480,38


In [120]:
print(f"Общая сумма продаж: {merged["total_cost"].sum():.2f}")
print(f"Среднее число рабочих дней доставки: {merged["business_days"].mean():.2f}")

Общая сумма продаж: 22221351.00
Среднее число рабочих дней доставки: 6.02
